In [1]:
import geopandas as gpd
import pandas as pd
import folium
import json
from shapely import wkt

def load_route_from_json(json_file_path):
    """
    Load route data from JSON and convert to GeoDataFrame
    """
    # Read JSON file
    with open(json_file_path, 'r') as f:
        data = json.load(f)
    
    # Extract route segments from the nested structure
    if 'route_segments' in data:
        segments_data = data['route_segments']
    else:
        segments_data = data
    
    # Convert to DataFrame
    df = pd.DataFrame(segments_data)
    
    # Convert WKT to geometry
    df['geometry'] = df['geometry_wkt'].apply(wkt.loads)
    
    # Convert to GeoDataFrame
    gdf = gpd.GeoDataFrame(df, geometry='geometry', crs="EPSG:4326")
    
    print(f"Loaded {len(gdf)} route segments")
    print(f"Total cost: {data.get('total_cost', 'N/A')}")
    
    return gdf

def create_route_map_with_layers(route_gdf, map_center=None, zoom_start=13):
    """
    Create a Folium map with multiple base layers (Satellite, Topo, OpenStreetMap)
    """
    # Calculate map center if not provided
    if map_center is None:
        centroid = route_gdf.geometry.unary_union.centroid
        map_center = [centroid.y, centroid.x]
    
    # Create base map with OpenStreetMap as default
    m = folium.Map(location=map_center, zoom_start=zoom_start, control_scale=True)
    
    # Add multiple tile layers
    # OpenStreetMap (default)
    folium.TileLayer(
        tiles='OpenStreetMap',
        name='OpenStreetMap',
        attr='OpenStreetMap contributors'
    ).add_to(m)
    
    # Satellite/Imagery layer (Esri World Imagery)
    folium.TileLayer(
        tiles='https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}',
        name='Satellite',
        attr='Tiles &copy; Esri &mdash; Source: Esri, i-cubed, USDA, USGS, AEX, GeoEye, Getmapping, Aerogrid, IGN, IGP, UPR-EGP, and the GIS User Community',
        overlay=False
    ).add_to(m)
    
    # Topographic layer (Esri World Topo Map)
    folium.TileLayer(
        tiles='https://server.arcgisonline.com/ArcGIS/rest/services/World_Topo_Map/MapServer/tile/{z}/{y}/{x}',
        name='Topographic',
        attr='Tiles &copy; Esri &mdash; Esri, DeLorme, NAVTEQ, TomTom, Intermap, iPC, USGS, FAO, NPS, NRCAN, GeoBase, Kadaster NL, Ordnance Survey, Esri Japan, METI, Esri China (Hong Kong), and the GIS User Community',
        overlay=False
    ).add_to(m)
    
    # Terrain layer (Stamen Terrain)
    folium.TileLayer(
        tiles='Stamen Terrain',
        name='Terrain',
        attr='Map tiles by <a href="http://stamen.com">Stamen Design</a>, under <a href="http://creativecommons.org/licenses/by/3.0">CC BY 3.0</a>. Data by <a href="http://openstreetmap.org">OpenStreetMap</a>, under <a href="http://www.openstreetmap.org/copyright">ODbL</a>.',
        overlay=False
    ).add_to(m)
    
    # Color palette for different sequence numbers
    colors = ['red', 'blue', 'green', 'purple', 'orange', 'darkred', 
              'lightred', 'beige', 'darkblue', 'darkgreen', 'cadetblue', 
              'darkpurple', 'white', 'pink', 'lightblue', 'lightgreen']
    
    # Create feature groups for better organization
    route_group = folium.FeatureGroup(name='Route Segments', show=True)
    points_group = folium.FeatureGroup(name='Points of Interest', show=True)
    
    # Add each route segment to the route group
    for idx, row in route_gdf.iterrows():
        # Get color based on sequence
        color = colors[row['seq'] % len(colors)]
        
        # Create popup content
        popup_text = f"""
        <div style="font-family: Arial, sans-serif;">
            <h4 style="margin: 0; color: {color};">Segment {row['seq']}</h4>
            <hr style="margin: 5px 0;">
            <b>Start Node:</b> {row['start_node_id']}<br>
            <b>End Node:</b> {row['end_node_id']}<br>
            <b>Leg Cost:</b> {row['leg_cost']:.2f}<br>
            <b>Cumulative Cost:</b> {row['cumulative_cost']:.2f}
        </div>
        """
        
        # Convert geometry to GeoJSON and add to map
        geojson_data = gpd.GeoSeries([row['geometry']]).__geo_interface__
        
        folium.GeoJson(
            geojson_data,
            style_function=lambda x, color=color: {
                'color': color,
                'weight': 6,
                'opacity': 0.8,
                'lineJoin': 'round'
            },
            popup=folium.Popup(popup_text, max_width=300)
        ).add_to(route_group)
    
    # Add start and end points to points group
    add_points_to_feature_group(route_gdf, points_group)
    
    # Add feature groups to map
    route_group.add_to(m)
    points_group.add_to(m)
    
    # Add layer control
    folium.LayerControl(collapsed=False).add_to(m)
    
    # Add a title/legend
    add_map_legend(m)
    
    return m

def add_points_to_feature_group(route_gdf, feature_group):
    """
    Add points to a specific feature group
    """
    points_added = set()
    
    for idx, row in route_gdf.iterrows():
        # Start point
        start_point = row.geometry.coords[0]
        start_coords = (start_point[1], start_point[0])
        
        if start_coords not in points_added:
            if idx == 0:
                folium.Marker(
                    location=[start_point[1], start_point[0]],
                    popup=folium.Popup(
                        f"<b>START</b><br>Node: {row['start_node_id']}<br>Segment: {row['seq']}", 
                        max_width=200
                    ),
                    icon=folium.Icon(color='green', icon='play', prefix='fa'),
                    tooltip="Start Point"
                ).add_to(feature_group)
            else:
                folium.Marker(
                    location=[start_point[1], start_point[0]],
                    popup=folium.Popup(
                        f"<b>Waypoint</b><br>Node: {row['start_node_id']}<br>Segment: {row['seq']}", 
                        max_width=200
                    ),
                    icon=folium.Icon(color='blue', icon='info-sign'),
                    tooltip=f"Waypoint {row['start_node_id']}"
                ).add_to(feature_group)
            points_added.add(start_coords)
        
        # End point
        end_point = row.geometry.coords[-1]
        end_coords = (end_point[1], end_point[0])
        
        if end_coords not in points_added:
            if idx == len(route_gdf) - 1:
                folium.Marker(
                    location=[end_point[1], end_point[0]],
                    popup=folium.Popup(
                        f"<b>END</b><br>Node: {row['end_node_id']}<br>Segment: {row['seq']}", 
                        max_width=200
                    ),
                    icon=folium.Icon(color='red', icon='stop', prefix='fa'),
                    tooltip="End Point"
                ).add_to(feature_group)
            else:
                folium.Marker(
                    location=[end_point[1], end_point[0]],
                    popup=folium.Popup(
                        f"<b>Waypoint</b><br>Node: {row['end_node_id']}<br>Segment: {row['seq']}", 
                        max_width=200
                    ),
                    icon=folium.Icon(color='blue', icon='info-sign'),
                    tooltip=f"Waypoint {row['end_node_id']}"
                ).add_to(feature_group)
            points_added.add(end_coords)

def add_map_legend(m):
    """
    Add a custom legend to the map
    """
    legend_html = '''
    <div style="position: fixed; 
                top: 10px; left: 50px; width: 200px; height: auto; 
                background-color: white; border:2px solid grey; z-index:9999; 
                font-size:14px; padding: 10px; border-radius: 5px;">
        <p style="margin: 0 0 5px 0;"><b>Route Legend</b></p>
        <p style="margin: 2px 0;"><span style="color: green;">●</span> Start Point</p>
        <p style="margin: 2px 0;"><span style="color: red;">●</span> End Point</p>
        <p style="margin: 2px 0;"><span style="color: blue;">●</span> Waypoints</p>
        <p style="margin: 2px 0;"><span style="color: red;">━━</span> Route Segments</p>
        <p style="margin: 5px 0 0 0; font-size: 12px; color: #666;">
            Use layer control (top-right) to switch base maps
        </p>
    </div>
    '''
    m.get_root().html.add_child(folium.Element(legend_html))

def create_satellite_focused_map(route_gdf, map_center=None, zoom_start=13):
    """
    Create a map with Satellite as the default base layer
    """
    if map_center is None:
        centroid = route_gdf.geometry.unary_union.centroid
        map_center = [centroid.y, centroid.x]
    
    # Create map with Satellite as default
    m = folium.Map(
        location=map_center, 
        zoom_start=zoom_start, 
        control_scale=True,
        tiles='https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}',
        attr='Esri World Imagery'
    )
    
    # Add other tile layers
    folium.TileLayer(
        tiles='OpenStreetMap',
        name='OpenStreetMap',
        attr='OpenStreetMap contributors'
    ).add_to(m)
    
    folium.TileLayer(
        tiles='https://server.arcgisonline.com/ArcGIS/rest/services/World_Topo_Map/MapServer/tile/{z}/{y}/{x}',
        name='Topographic',
        attr='Esri World Topo Map'
    ).add_to(m)
    
    folium.TileLayer(
        tiles='Stamen Terrain',
        name='Terrain',
        attr='Stamen Terrain'
    ).add_to(m)
    
    # Add route data
    colors = ['red', 'blue', 'green', 'purple', 'orange', 'darkred', 
              'lightred', 'beige', 'darkblue', 'darkgreen']
    
    for idx, row in route_gdf.iterrows():
        color = colors[row['seq'] % len(colors)]
        
        popup_text = f"""
        <b>Segment {row['seq']}</b><br>
        Start: Node {row['start_node_id']}<br>
        End: Node {row['end_node_id']}<br>
        Leg Cost: {row['leg_cost']:.2f}<br>
        Cumulative: {row['cumulative_cost']:.2f}
        """
        
        geojson_data = gpd.GeoSeries([row['geometry']]).__geo_interface__
        
        folium.GeoJson(
            geojson_data,
            style_function=lambda x, color=color: {
                'color': color,
                'weight': 8,  # Thicker lines for better visibility on satellite
                'opacity': 0.9,
                'lineJoin': 'round'
            },
            popup=folium.Popup(popup_text, max_width=300)
        ).add_to(m)
    
    # Add points
    add_route_points(m, route_gdf)
    folium.LayerControl(collapsed=False).add_to(m)
    add_map_legend(m)
    
    return m

def add_route_points(m, route_gdf):
    """
    Add points directly to map (for simple maps)
    """
    points_added = set()
    
    for idx, row in route_gdf.iterrows():
        # Start point
        start_point = row.geometry.coords[0]
        start_coords = (start_point[1], start_point[0])
        
        if start_coords not in points_added:
            icon_color = 'green' if idx == 0 else 'blue'
            icon_type = 'play' if idx == 0 else 'info-sign'
            point_type = 'START' if idx == 0 else 'Waypoint'
            
            folium.Marker(
                location=[start_point[1], start_point[0]],
                popup=folium.Popup(
                    f"<b>{point_type}</b><br>Node: {row['start_node_id']}<br>Segment: {row['seq']}", 
                    max_width=200
                ),
                icon=folium.Icon(color=icon_color, icon=icon_type, prefix='fa' if idx == 0 else None),
                tooltip=f"{point_type} Node {row['start_node_id']}"
            ).add_to(m)
            points_added.add(start_coords)
        
        # End point
        end_point = row.geometry.coords[-1]
        end_coords = (end_point[1], end_point[0])
        
        if end_coords not in points_added:
            icon_color = 'red' if idx == len(route_gdf) - 1 else 'blue'
            icon_type = 'stop' if idx == len(route_gdf) - 1 else 'info-sign'
            point_type = 'END' if idx == len(route_gdf) - 1 else 'Waypoint'
            
            folium.Marker(
                location=[end_point[1], end_point[0]],
                popup=folium.Popup(
                    f"<b>{point_type}</b><br>Node: {row['end_node_id']}<br>Segment: {row['seq']}", 
                    max_width=200
                ),
                icon=folium.Icon(color=icon_color, icon=icon_type, prefix='fa' if idx == len(route_gdf) - 1 else None),
                tooltip=f"{point_type} Node {row['end_node_id']}"
            ).add_to(m)
            points_added.add(end_coords)

def analyze_route(route_gdf):
    """
    Analyze and print route statistics
    """
    print("=== ROUTE ANALYSIS ===")
    print(f"Total segments: {len(route_gdf)}")
    print(f"Total cost: {route_gdf['cumulative_cost'].iloc[-1]:.2f}")
    
    # Calculate approximate total distance
    route_gdf_projected = route_gdf.to_crs('EPSG:3857')  # Web Mercator for distance calculation
    total_distance_km = route_gdf_projected.length.sum() / 1000
    print(f"Total distance: {total_distance_km:.2f} km")
    
    print("\nSegment details:")
    for idx, row in route_gdf.iterrows():
        segment_length_km = route_gdf_projected.iloc[idx].geometry.length / 1000
        print(f"Segment {row['seq']}: "
              f"Node {row['start_node_id']} → Node {row['end_node_id']} "
              f"(cost: {row['leg_cost']:.2f}, "
              f"distance: {segment_length_km:.2f} km)")

# Main execution
def main():
    # Load your route data
    route_gdf = load_route_from_json('../datasets/processed/example_route.json')
    
    # Analyze the route
    analyze_route(route_gdf)
    
    # Create map with multiple layers
    print("\nCreating multi-layer map...")
    multi_layer_map = create_route_map_with_layers(route_gdf)
    multi_layer_map.save('../datasets/processed/route_map_multi_layer.html')
    
    # Create satellite-focused map
    print("Creating satellite-focused map...")
    satellite_map = create_satellite_focused_map(route_gdf)
    satellite_map.save('../datasets/processed/route_map_satellite.html')
    
    print("\nMaps saved as:")
    print("- route_map_multi_layer.html (OpenStreetMap default with layer control)")
    print("- route_map_satellite.html (Satellite default with layer control)")
    print("\nOpen these files in your web browser to view the routes!")
    print("Use the layer control (top-right) to switch between map types!")

# For Jupyter Notebook usage:
def display_in_notebook():
    """
    Use this function in Jupyter notebooks
    """
    route_gdf = load_route_from_json('../datasets/processed/example_route.json')
    analyze_route(route_gdf)
    
    # Display the multi-layer map in notebook
    return create_route_map_with_layers(route_gdf)

if __name__ == "__main__":
    main()

Loaded 16 route segments
Total cost: 347.7570291003674
=== ROUTE ANALYSIS ===
Total segments: 16
Total cost: 347.76
Total distance: 39.64 km

Segment details:
Segment 1: Node 1 → Node 2 (cost: 22.46, distance: 21.49 km)
Segment 1: Node 1 → Node 2 (cost: 19.17, distance: 0.98 km)
Segment 1: Node 1 → Node 2 (cost: 50.56, distance: 0.30 km)
Segment 1: Node 1 → Node 2 (cost: 48.68, distance: 0.31 km)
Segment 1: Node 1 → Node 2 (cost: 21.54, distance: 0.62 km)
Segment 1: Node 1 → Node 2 (cost: 22.29, distance: 0.08 km)
Segment 1: Node 1 → Node 2 (cost: 2.45, distance: 11.14 km)
Segment 1: Node 1 → Node 2 (cost: 15.71, distance: 0.20 km)
Segment 1: Node 1 → Node 2 (cost: 16.11, distance: 0.27 km)
Segment 1: Node 1 → Node 2 (cost: 20.34, distance: 1.01 km)
Segment 1: Node 1 → Node 2 (cost: 22.38, distance: 0.06 km)
Segment 1: Node 1 → Node 2 (cost: 16.64, distance: 0.81 km)
Segment 1: Node 1 → Node 2 (cost: 19.05, distance: 0.04 km)
Segment 1: Node 1 → Node 2 (cost: 0.00, distance: 1.89 km)
S

/var/folders/tf/9qxxbg0d70qd569tqtkgf6100000gn/T/ipykernel_48245/3242086569.py:41: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  centroid = route_gdf.geometry.unary_union.centroid
/var/folders/tf/9qxxbg0d70qd569tqtkgf6100000gn/T/ipykernel_48245/3242086569.py:221: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  centroid = route_gdf.geometry.unary_union.centroid


In [2]:
interactive_map = display_in_notebook()
interactive_map

Loaded 16 route segments
Total cost: 347.7570291003674
=== ROUTE ANALYSIS ===
Total segments: 16
Total cost: 347.76
Total distance: 39.64 km

Segment details:
Segment 1: Node 1 → Node 2 (cost: 22.46, distance: 21.49 km)
Segment 1: Node 1 → Node 2 (cost: 19.17, distance: 0.98 km)
Segment 1: Node 1 → Node 2 (cost: 50.56, distance: 0.30 km)
Segment 1: Node 1 → Node 2 (cost: 48.68, distance: 0.31 km)
Segment 1: Node 1 → Node 2 (cost: 21.54, distance: 0.62 km)
Segment 1: Node 1 → Node 2 (cost: 22.29, distance: 0.08 km)
Segment 1: Node 1 → Node 2 (cost: 2.45, distance: 11.14 km)
Segment 1: Node 1 → Node 2 (cost: 15.71, distance: 0.20 km)
Segment 1: Node 1 → Node 2 (cost: 16.11, distance: 0.27 km)
Segment 1: Node 1 → Node 2 (cost: 20.34, distance: 1.01 km)
Segment 1: Node 1 → Node 2 (cost: 22.38, distance: 0.06 km)
Segment 1: Node 1 → Node 2 (cost: 16.64, distance: 0.81 km)
Segment 1: Node 1 → Node 2 (cost: 19.05, distance: 0.04 km)
Segment 1: Node 1 → Node 2 (cost: 0.00, distance: 1.89 km)
S

/var/folders/tf/9qxxbg0d70qd569tqtkgf6100000gn/T/ipykernel_48245/3242086569.py:41: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  centroid = route_gdf.geometry.unary_union.centroid
